In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. MODEL LOADING (Gemma 3 1B-IT)
# ══════════════════════════════════════════════════════════════════════════════

import sys
import time
import torch
import threading
import numpy as np
from transformers import AutoTokenizer, TextIteratorStreamer
from transformers.models.gemma3 import Gemma3ForCausalLM
import os
import sys
import subprocess
import time
import torch
from threading import Thread

# Optimize Torch for CPU (if consistent with previous notebook)
torch.set_num_threads(os.cpu_count())
torch.set_num_interop_threads(min(os.cpu_count(), 4))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Update this path to your local model folder
GEMMA_PATH = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\kaggle_models\gemma-3-1b-it-v1"
# NOTE: If the model is not in this path, please update GEMMA_PATH

print(f"Loading model from: {GEMMA_PATH}...")

try:
    tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH)
    model = Gemma3ForCausalLM.from_pretrained(
        GEMMA_PATH,
        torch_dtype=torch.float16,    # Requested Precision
        attn_implementation="sdpa",   # Requested Attention
    	device_map="auto"             # Auto-detect (CPU/GPU)
    )
    print("✅ Model loaded successfully!")
    print(f"  Device: {model.device}")
    print(f"  Dtype: {model.dtype}")

except OSError:
    print(f"❌ Could not find model at {GEMMA_PATH}")
    print("Please download 'google/gemma-3-1b-it' from Hugging Face and update the path.")

C:\Users\divyam.tewary\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0+cpu
CUDA Available: False
Loading model from: C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\kaggle_models\gemma-3-1b-it-v1...


Loading weights: 100%|██████████| 340/340 [00:01<00:00, 258.29it/s, Materializing param=model.norm.weight]                                


✅ Model loaded successfully!
  Device: cpu
  Dtype: torch.float16


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. INTERACTIVE CHAT LOOP (CLI-Style) - v3 Robust Threading
# ══════════════════════════════════════════════════════════════════════════════

class SimpleChat:
    def __init__(self, system_prompt=None):
        self.history = []
        if system_prompt:
             self.history.append({"role": "user", "content": system_prompt})
             self.history.append({"role": "model", "content": "Understood. I am ready."})
            
    def chat_loop(self):
        print("\n💬 GEMMA 3 1B-IT CHAT (Type 'exit' to stop, 'clear' to reset)")
        print("────────────────────────────────────────────────────────")
        
        while True:
            try:
                user_input = input("\n👤 YOU: ").strip()
                if user_input.lower() in ["exit", "quit"]:
                    print("👋 Exiting chat...")
                    break
                if user_input.lower() == "clear":
                    self.history = []
                    print("🧹 Memory cleared.")
                    continue
                if not user_input:
                    continue
            except EOFError:
                break
                
            self.history.append({"role": "user", "content": user_input})
            
            try:
                prompt = tokenizer.apply_chat_template(
                    self.history, 
                    tokenize=False, 
                    add_generation_prompt=True
                )
            except Exception as e:
                print(f"⚠️ Template Error: {e}")
                prompt = f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            print(f"  (Context window: {inputs.input_ids.shape[1]} tokens)")

            print("🤖 MODEL: ", end="", flush=True)
            
            # REMOVED TIMEOUT: Let it wait as long as CPU needs for prefill
            streamer = TextIteratorStreamer(
                tokenizer, 
                timeout=None,    
                skip_prompt=True, 
                skip_special_tokens=True 
            )
            
            terminators = [
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<end_of_turn>")
            ]
            
            generation_kwargs = dict(
                inputs,
                streamer=streamer,
                max_new_tokens=512,
                do_sample=False,
                eos_token_id=terminators,
                pad_token_id=tokenizer.eos_token_id
            )
            
            # Wrapper to catch errors inside the thread
            def generate_safely():
                try:
                    model.generate(**generation_kwargs)
                except Exception as e:
                    print(f"\n❌ GENERATION THREAD CRASHED: {e}")

            thread = Thread(target=generate_safely)
            thread.start()
            
            full_response = ""
            try:
                for new_text in streamer:
                    if "<end_of_turn>" in new_text:
                        new_text = new_text.replace("<end_of_turn>", "")
                        print(new_text, end="", flush=True)
                        full_response += new_text
                        break
                    print(new_text, end="", flush=True)
                    full_response += new_text
            except Exception as e:
                print(f"\n⚠️ Stream Error: {e}")
                
            print() 
            self.history.append({"role": "model", "content": full_response})

# ─── Launch Chat ───
chat_session = SimpleChat(
    system_prompt="You are a helpful and concise AI assistant. Answer directly."
)



### UI_V9

In [19]:
# ══════════════════════════════════════════════════════════════════════════════

# GEMMA CHAT v12 — LUXE COPILOT WORKSPACE

# - Premium dark workspace with prominent IndiGo branding

# - Model / Persona / System Prompt selectors

# - Collapsible sections, fluid panel toggles, dynamic chat bubbles

# ══════════════════════════════════════════════════════════════════════════════



import os

import re

from threading import Thread



from PyQt5.QtCore import Qt, QSize, QThread, pyqtSignal, QTimer, QPropertyAnimation, QEasingCurve

from PyQt5.QtGui import QFont, QFontDatabase, QIcon, QPixmap, QTextOption, QColor

from PyQt5.QtWidgets import (

    QApplication, QMainWindow, QWidget, QFrame, QLabel, QPushButton, QLineEdit,

    QTextBrowser, QTextEdit, QListWidget, QListWidgetItem, QFileDialog, QMessageBox,

    QHBoxLayout, QVBoxLayout, QSizePolicy, QScrollArea, QStackedWidget,

    QGraphicsOpacityEffect, QGraphicsDropShadowEffect, QComboBox

    )



from transformers import TextIteratorStreamer



# ── THEME ─────────────────────────────────────────────────────────────────────

CLR_BG = "#070B16"

CLR_SIDEBAR = "#0B1328"

CLR_MAIN = "#0D172D"

CLR_PANEL = "#121F3B"

CLR_PANEL_2 = "#1A2950"

CLR_HOVER = "#23345F"

CLR_BORDER = "#2D3F6A"

CLR_TEXT = "#E7EEFF"

CLR_MUTED = "#9BB1DA"

CLR_ACCENT = "#4A87FF"

CLR_ACCENT_2 = "#6AA8FF"

CLR_USER = "#2E63D9"

CLR_MODEL = "#1B2A4D"

LOGO_PATH = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\IndiGo_logo.png"



MODEL_CHOICES = [

    "Gemma 3 1B-IT (Loaded)",

    "Gemma 3 4B-IT (Design Placeholder)",

    "Gemma 3 12B-IT (Design Placeholder)",

]



PERSONA_CHOICES = {

    "Balanced Assistant": "Be accurate, concise, and practical.",

    "Coder Pro": "You are an elite coding copilot. Return production-ready code with concise rationale.",

    "Analyst": "You are a rigorous analyst. Structure findings clearly with assumptions and trade-offs.",

    "Creative Partner": "You are imaginative and polished while staying grounded in user goals.",

}



SYSTEM_PROMPTS = {

    "Default": "You are a helpful and concise AI assistant. Format clearly using headings and bullet points where useful.",

    "Strict Technical": "You are a strict technical assistant. Keep precision high, avoid fluff, and state assumptions.",

    "Executive Summary": "Respond in executive-ready format with clear sections, key points, and action items.",

    "Tutor Mode": "Teach clearly with examples, step-by-step logic, and short checkpoints.",

}



def _preferred_font():

    families = set(QFontDatabase().families())

    for name in ["Segoe UI Variable", "Inter", "Segoe UI", "Arial"]:

        if name in families:

            return QFont(name, 10)

    return QFont("Sans Serif", 10)



class HoverButton(QPushButton):

    def __init__(self, text, click_handler=None, min_h=36, compact=False, accent=False):

        super().__init__(text)

        self.setCursor(Qt.PointingHandCursor)

        self.setMinimumHeight(min_h)

        pad = "6px 10px" if compact else "8px 12px"

        if accent:

            self.setStyleSheet(

                f"""

                QPushButton {{

                    background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 {CLR_ACCENT}, stop:1 {CLR_ACCENT_2});

                    color: white;

                    border: 0;

                    border-radius: 11px;

                    padding: {pad};

                    font-weight: 700;

                }}

                QPushButton:hover {{ background: {CLR_ACCENT_2}; }}

                QPushButton:pressed {{ background: {CLR_ACCENT}; }}

                """

            )

        else:

            self.setStyleSheet(

                f"""

                QPushButton {{

                    background-color: {CLR_PANEL};

                    color: {CLR_TEXT};

                    border: 1px solid {CLR_BORDER};

                    border-radius: 10px;

                    padding: {pad};

                    text-align: left;

                    font-weight: 600;

                }}

                QPushButton:hover {{ background-color: {CLR_HOVER}; border: 1px solid {CLR_MUTED}; }}

                QPushButton:pressed {{ background-color: {CLR_PANEL_2}; }}

                """

            )

        if click_handler:

            self.clicked.connect(click_handler)



class ChatBubble(QFrame):

    def __init__(self, role, text=""):

        super().__init__()

        self.role = role

        self.text = text

        self.setObjectName("bubble")



        bubble_grad = (

            "qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #2D66DE, stop:1 #1E4FBF)"

            if role == "user"

            else "qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #16243F, stop:1 #22355C)"

        )



        self.setStyleSheet(

            f"""

            QFrame#bubble {{

                background: {bubble_grad};

                border: 1px solid {CLR_BORDER};

                border-radius: 14px;

            }}

            QLabel {{ color: {CLR_TEXT}; }}

            QTextBrowser {{

                background: transparent;

                border: none;

                color: {CLR_TEXT};

                font-size: 14px;

            }}

            """

        )



        shadow = QGraphicsDropShadowEffect(self)

        shadow.setBlurRadius(22)

        shadow.setOffset(0, 4)

        shadow.setColor(QColor(0, 0, 0, 110))

        self.setGraphicsEffect(shadow)

        self.setSizePolicy(QSizePolicy.Preferred, QSizePolicy.Minimum)



        lay = QVBoxLayout(self)

        lay.setContentsMargins(12, 8, 12, 10)

        lay.setSpacing(6)



        role_lbl = QLabel("You" if role == "user" else "IndiGo AI")

        role_lbl.setStyleSheet(f"color: {CLR_MUTED}; font-weight: 700;")

        lay.addWidget(role_lbl)



        self.body = QTextBrowser()

        self.body.setOpenExternalLinks(True)

        self.body.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Minimum)

        self.body.setMaximumHeight(620)

        self.body.setVerticalScrollBarPolicy(Qt.ScrollBarAlwaysOff)

        self.body.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)

        wrap = self.body.document().defaultTextOption()

        wrap.setWrapMode(QTextOption.WrapAtWordBoundaryOrAnywhere)

        self.body.document().setDefaultTextOption(wrap)

        self.body.setMarkdown(self._to_markdown(text))

        lay.addWidget(self.body)



    def _to_markdown(self, raw):

        cleaned = raw.replace("\r\n", "\n")

        cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()

        return cleaned if cleaned else " "



    def update_text(self, text):

        self.text = text

        self.body.setMarkdown(self._to_markdown(text))

        self.body.document().adjustSize()

        h = int(self.body.document().size().height()) + 26

        self.body.setMinimumHeight(min(max(h, 46), 600))



class GenerationWorker(QThread):

    chunk = pyqtSignal(str)

    finished_ok = pyqtSignal()

    failed = pyqtSignal(str)



    def __init__(self, mdl, tok, prompt_history):

        super().__init__()

        self.model = mdl

        self.tok = tok

        self.prompt_history = prompt_history



    def run(self):

        try:

            prompt = self.tok.apply_chat_template(

                self.prompt_history, tokenize=False, add_generation_prompt=True

            )

            inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)

            streamer = TextIteratorStreamer(

                self.tok, timeout=None, skip_prompt=True, skip_special_tokens=True

            )

            terminators = [self.tok.eos_token_id, self.tok.convert_tokens_to_ids("<end_of_turn>")]



            def _generate():

                self.model.generate(

                    **inputs,

                    streamer=streamer,

                    max_new_tokens=512,

                    do_sample=False,

                    eos_token_id=terminators,

                    pad_token_id=self.tok.eos_token_id,

                )



            t = Thread(target=_generate, daemon=True)

            t.start()



            for token in streamer:

                token = token.replace("<end_of_turn>", "")

                if token:

                    self.chunk.emit(token)



            t.join()

            self.finished_ok.emit()

        except Exception as e:

            self.failed.emit(str(e))



class PremiumGemmaWindow(QMainWindow):

    LEFT_W = 300

    RIGHT_W = 360



    def __init__(self, mdl, tok):

        super().__init__()

        self.model = mdl

        self.tok = tok



        self.worker = None

        self.current_reply = ""

        self.current_bubble = None



        self.sessions = []

        self.current_session_index = -1

        self.files_content = {}



        self.left_expanded = True

        self.right_expanded = True



        self.setWindowTitle("IndiGo AI Workspace")

        self.resize(1540, 950)

        self.setStyleSheet(f"background-color: {CLR_BG}; color: {CLR_TEXT};")

        QApplication.instance().setFont(_preferred_font())



        self.stack = QStackedWidget()

        self.setCentralWidget(self.stack)



        self._build_intro_page()

        self._build_workspace_page()



        self.stack.addWidget(self.intro_page)

        self.stack.addWidget(self.workspace_page)

        self.stack.setCurrentWidget(self.intro_page)



        self.statusBar().setStyleSheet(f"background:{CLR_SIDEBAR}; color:{CLR_MUTED}; border-top:1px solid {CLR_BORDER};")

        self.statusBar().showMessage("Ready")

        self._apply_windows_dark_titlebar()



        self._create_new_session(initial=True)

        QTimer.singleShot(800, self.start_intro_transition)



    # ── Intro ─────────────────────────────────────────────────────────────────

    def _build_intro_page(self):

        self.intro_page = QWidget()

        self.intro_page.setStyleSheet(

            "background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #060B16, stop:1 #102040);"

        )



        lay = QVBoxLayout(self.intro_page)

        lay.setContentsMargins(80, 80, 80, 80)

        lay.setSpacing(16)

        lay.addStretch(1)



        logo = QLabel()

        logo.setAlignment(Qt.AlignCenter)

        if os.path.exists(LOGO_PATH):

            pix = QPixmap(LOGO_PATH)

            logo.setPixmap(pix.scaled(240, 78, Qt.KeepAspectRatio, Qt.SmoothTransformation))

        else:

            logo.setText("INDIGO")

            logo.setStyleSheet(f"font-size:36px; font-weight:900; color:{CLR_TEXT};")



        title = QLabel("Premium AI Copilot Workspace")

        title.setAlignment(Qt.AlignCenter)

        title.setStyleSheet(f"font-size: 42px; font-weight: 800; color: {CLR_TEXT};")



        subtitle = QLabel("Model-aware chat with persona and system controls.")

        subtitle.setAlignment(Qt.AlignCenter)

        subtitle.setStyleSheet(f"font-size: 18px; color: {CLR_MUTED};")



        start_btn = HoverButton("Launch Workspace", self.start_intro_transition, min_h=46, accent=True)

        start_btn.setFixedWidth(220)



        btn_wrap = QHBoxLayout()

        btn_wrap.addStretch(1)

        btn_wrap.addWidget(start_btn)

        btn_wrap.addStretch(1)



        lay.addWidget(logo)

        lay.addWidget(title)

        lay.addWidget(subtitle)

        lay.addLayout(btn_wrap)

        lay.addStretch(1)



    def start_intro_transition(self):

        if self.stack.currentWidget() is self.workspace_page:

            return

        effect = QGraphicsOpacityEffect(self.intro_page)

        self.intro_page.setGraphicsEffect(effect)

        self.fade_anim = QPropertyAnimation(effect, b"opacity")

        self.fade_anim.setDuration(420)

        self.fade_anim.setStartValue(1.0)

        self.fade_anim.setEndValue(0.0)

        self.fade_anim.setEasingCurve(QEasingCurve.InOutQuad)

        self.fade_anim.finished.connect(self._show_workspace)

        self.fade_anim.start()



    def _show_workspace(self):

        self.stack.setCurrentWidget(self.workspace_page)

        self.statusBar().showMessage("Workspace ready", 1600)



    # ── Workspace ─────────────────────────────────────────────────────────────

    def _build_workspace_page(self):

        self.workspace_page = QWidget()

        self.workspace_page.setStyleSheet(

            "background: qlineargradient(x1:0,y1:0,x2:1,y2:1, stop:0 #070B16, stop:1 #0E1B35);"

        )

        root = QHBoxLayout(self.workspace_page)

        root.setContentsMargins(0, 0, 0, 0)

        root.setSpacing(0)



        self._build_left_panel()

        self._build_center_panel()

        self._build_right_panel()



        root.addWidget(self.left_panel)

        root.addWidget(self.center_panel, 1)

        root.addWidget(self.right_panel)



    def _build_logo_widget(self):

        w = QWidget()

        lay = QHBoxLayout(w)

        lay.setContentsMargins(4, 4, 4, 4)

        lay.setSpacing(10)



        icon = QLabel()

        icon.setFixedSize(120, 38)

        if os.path.exists(LOGO_PATH):

            pix = QPixmap(LOGO_PATH)

            icon.setPixmap(pix.scaled(118, 36, Qt.KeepAspectRatio, Qt.SmoothTransformation))

        else:

            icon.setText("INDIGO")

            icon.setStyleSheet(f"font-weight:900; color:{CLR_TEXT};")



        sub = QLabel("AI Command Center")

        sub.setStyleSheet(f"font-size:12px; font-weight:700; color:{CLR_MUTED};")



        lay.addWidget(icon)

        lay.addWidget(sub, 1)

        return w



    def _build_section_header(self, title, on_toggle):

        btn = QPushButton(f"▾  {title}")

        btn.setCursor(Qt.PointingHandCursor)

        btn.setStyleSheet(

            f"QPushButton {{ background: transparent; color:{CLR_MUTED}; border:none; text-align:left; font-weight:800; padding:4px 2px; }}"

        )

        btn.clicked.connect(on_toggle)

        return btn



    def _build_left_panel(self):

        self.left_panel = QFrame()

        self.left_panel.setMinimumWidth(0)

        self.left_panel.setMaximumWidth(self.LEFT_W)

        self.left_panel.setStyleSheet(

            f"background: qlineargradient(x1:0,y1:0,x2:0,y2:1, stop:0 {CLR_SIDEBAR}, stop:1 #0D1630); border-right:1px solid {CLR_BORDER};"

        )



        lay = QVBoxLayout(self.left_panel)

        lay.setContentsMargins(12, 12, 12, 12)

        lay.setSpacing(10)



        top_row = QHBoxLayout()

        self.btn_left_toggle = HoverButton("☰", self.on_toggle_left_panel, min_h=34, compact=True)

        self.btn_left_toggle.setFixedWidth(44)

        self.btn_home = HoverButton("Home", self.on_home_clicked, min_h=34, compact=True)

        if os.path.exists(LOGO_PATH):

            self.btn_home.setText("")

            self.btn_home.setIcon(QIcon(LOGO_PATH))

            self.btn_home.setIconSize(QSize(100, 28))

        top_row.addWidget(self.btn_left_toggle)

        top_row.addWidget(self.btn_home, 1)



        self.brand_card = QFrame()

        self.brand_card.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        bc = QVBoxLayout(self.brand_card)

        bc.setContentsMargins(12, 10, 12, 10)

        bc.addWidget(self._build_logo_widget())



        self.btn_new_chat = HoverButton("＋ New Conversation", self.on_new_chat_clicked, min_h=36, accent=True)



        self.section_sessions_btn = self._build_section_header("Conversations", self.on_toggle_sessions_section)

        self.sessions_box = QWidget()

        sessions_lay = QVBoxLayout(self.sessions_box)

        sessions_lay.setContentsMargins(0, 0, 0, 0)

        sessions_lay.setSpacing(6)

        self.session_list = QListWidget()

        self.session_list.setStyleSheet(

            f"""

            QListWidget {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:4px; }}

            QListWidget::item {{ padding:8px; border-radius:8px; }}

            QListWidget::item:hover {{ background:{CLR_PANEL}; }}

            QListWidget::item:selected {{ background:{CLR_HOVER}; }}

            """

        )

        self.session_list.itemClicked.connect(self.on_session_selected)

        sessions_lay.addWidget(self.session_list)



        lay.addLayout(top_row)

        lay.addWidget(self.brand_card)

        lay.addWidget(self.btn_new_chat)

        lay.addWidget(self.section_sessions_btn)

        lay.addWidget(self.sessions_box, 1)



    def _build_center_panel(self):

        self.center_panel = QFrame()

        self.center_panel.setStyleSheet("background: transparent;")

        lay = QVBoxLayout(self.center_panel)

        lay.setContentsMargins(14, 12, 14, 12)

        lay.setSpacing(10)



        header = QFrame()

        header.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        h = QHBoxLayout(header)

        h.setContentsMargins(10, 8, 10, 8)

        h.setSpacing(8)



        self.btn_left_toggle_header = HoverButton("☰", self.on_toggle_left_panel, min_h=34, compact=True)

        self.btn_left_toggle_header.setFixedWidth(44)



        title_wrap = QVBoxLayout()

        title_wrap.setContentsMargins(0, 0, 0, 0)

        title_wrap.setSpacing(0)

        self.title_lbl = QLabel("New Conversation")

        self.title_lbl.setStyleSheet(f"font-size:19px; font-weight:800; color:{CLR_TEXT};")

        self.subtitle_lbl = QLabel("Luxe Workspace")

        self.subtitle_lbl.setStyleSheet(f"font-size:12px; font-weight:600; color:{CLR_MUTED};")

        title_wrap.addWidget(self.title_lbl)

        title_wrap.addWidget(self.subtitle_lbl)



        self.cmb_model = QComboBox()

        self.cmb_model.addItems(MODEL_CHOICES)

        self.cmb_model.currentIndexChanged.connect(self.on_model_changed)

        self.cmb_model.setStyleSheet(self._combo_style())



        self.cmb_persona = QComboBox()

        self.cmb_persona.addItems(list(PERSONA_CHOICES.keys()))

        self.cmb_persona.setStyleSheet(self._combo_style())



        self.cmb_system_prompt = QComboBox()

        self.cmb_system_prompt.addItems(list(SYSTEM_PROMPTS.keys()))

        self.cmb_system_prompt.setStyleSheet(self._combo_style())



        self.btn_toggle_right = HoverButton("Context", self.on_toggle_right_panel, min_h=34, compact=True)

        self.btn_settings = HoverButton("Settings", self.on_settings_clicked, min_h=34, compact=True)



        h.addWidget(self.btn_left_toggle_header)

        h.addLayout(title_wrap)

        h.addStretch(1)

        h.addWidget(self.cmb_model)

        h.addWidget(self.cmb_persona)

        h.addWidget(self.cmb_system_prompt)

        h.addWidget(self.btn_toggle_right)

        h.addWidget(self.btn_settings)



        self.chat_scroll = QScrollArea()

        self.chat_scroll.setWidgetResizable(True)

        self.chat_scroll.setStyleSheet(

            f"QScrollArea {{ border:1px solid {CLR_BORDER}; border-radius:14px; background:{CLR_MAIN}; }}"

        )

        self.chat_host = QWidget()

        self.chat_vbox = QVBoxLayout(self.chat_host)

        self.chat_vbox.setContentsMargins(14, 14, 14, 14)

        self.chat_vbox.setSpacing(12)

        self.chat_vbox.addStretch(1)

        self.chat_scroll.setWidget(self.chat_host)



        input_wrap = QFrame()

        input_wrap.setStyleSheet(

            f"QFrame {{ background:{CLR_PANEL}; border:1px solid {CLR_BORDER}; border-radius:14px; }}"

        )

        i = QHBoxLayout(input_wrap)

        i.setContentsMargins(10, 8, 10, 8)

        i.setSpacing(8)



        self.input_box = QLineEdit()

        self.input_box.setPlaceholderText("Ask anything — code, analysis, planning, docs...")

        self.input_box.setStyleSheet(

            f"QLineEdit {{ background:transparent; border:none; color:{CLR_TEXT}; font-size:14px; }}"

        )

        self.input_box.returnPressed.connect(self.on_send_clicked)



        self.btn_upload = HoverButton("Upload", self.on_upload_clicked, min_h=34, compact=True)

        self.btn_send = HoverButton("Send", self.on_send_clicked, min_h=34, compact=True, accent=True)

        self.btn_send.setFixedWidth(92)



        i.addWidget(self.input_box, 1)

        i.addWidget(self.btn_upload)

        i.addWidget(self.btn_send)



        lay.addWidget(header)

        lay.addWidget(self.chat_scroll, 1)

        lay.addWidget(input_wrap)



    def _build_right_panel(self):

        self.right_panel = QFrame()

        self.right_panel.setMinimumWidth(0)

        self.right_panel.setMaximumWidth(self.RIGHT_W)

        self.right_panel.setStyleSheet(

            f"background: qlineargradient(x1:0,y1:0,x2:0,y2:1, stop:0 #0C1530, stop:1 #101F3E); border-left:1px solid {CLR_BORDER};"

        )



        lay = QVBoxLayout(self.right_panel)

        lay.setContentsMargins(12, 12, 12, 12)

        lay.setSpacing(10)



        row = QHBoxLayout()

        lbl = QLabel("Context & Controls")

        lbl.setStyleSheet(f"font-size:14px; font-weight:800; color:{CLR_TEXT};")

        self.btn_right_toggle = HoverButton("⇤", self.on_toggle_right_panel, min_h=34, compact=True)

        self.btn_right_toggle.setFixedWidth(44)

        row.addWidget(lbl)

        row.addStretch(1)

        row.addWidget(self.btn_right_toggle)



        self.section_files_btn = self._build_section_header("Uploaded Files", self.on_toggle_files_section)

        self.files_box = QWidget()

        files_lay = QVBoxLayout(self.files_box)

        files_lay.setContentsMargins(0, 0, 0, 0)

        files_lay.setSpacing(6)

        self.file_list = QListWidget()

        self.file_list.setStyleSheet(

            f"""

            QListWidget {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:4px; }}

            QListWidget::item {{ padding:8px; border-radius:8px; }}

            QListWidget::item:hover {{ background:{CLR_PANEL}; }}

            QListWidget::item:selected {{ background:{CLR_HOVER}; }}

            """

        )

        self.file_list.itemClicked.connect(self.on_file_selected)

        self.file_preview = QTextEdit()

        self.file_preview.setReadOnly(True)

        self.file_preview.setPlaceholderText("File preview appears here")

        self.file_preview.setStyleSheet(

            f"QTextEdit {{ background:{CLR_MAIN}; color:{CLR_TEXT}; border:1px solid {CLR_BORDER}; border-radius:10px; padding:8px; }}"

        )

        files_lay.addWidget(self.file_list)

        files_lay.addWidget(self.file_preview)



        self.section_controls_btn = self._build_section_header("Runtime Controls", self.on_toggle_controls_section)

        self.controls_box = QWidget()

        controls_lay = QVBoxLayout(self.controls_box)

        controls_lay.setContentsMargins(0, 0, 0, 0)

        controls_lay.setSpacing(8)

        self.info_model = QLabel("Model: Gemma 3 1B-IT")

        self.info_model.setStyleSheet(f"color:{CLR_MUTED};")

        self.info_persona = QLabel("Persona: Balanced Assistant")

        self.info_persona.setStyleSheet(f"color:{CLR_MUTED};")

        self.info_prompt = QLabel("System: Default")

        self.info_prompt.setStyleSheet(f"color:{CLR_MUTED};")

        controls_lay.addWidget(self.info_model)

        controls_lay.addWidget(self.info_persona)

        controls_lay.addWidget(self.info_prompt)



        lay.addLayout(row)

        lay.addWidget(self.section_files_btn)

        lay.addWidget(self.files_box, 1)

        lay.addWidget(self.section_controls_btn)

        lay.addWidget(self.controls_box)



    # ── Interactions ───────────────────────────────────────────────────────────

    def _combo_style(self):

        return (

            f"""

            QComboBox {{

                background:{CLR_MAIN};

                color:{CLR_TEXT};

                border:1px solid {CLR_BORDER};

                border-radius:10px;

                padding:6px 10px;

                min-width:160px;

                font-weight:600;

            }}

            QComboBox QAbstractItemView {{

                background:{CLR_MAIN};

                color:{CLR_TEXT};

                border:1px solid {CLR_BORDER};

                selection-background-color:{CLR_HOVER};

            }}

            """

        )



    def on_home_clicked(self):

        self.stack.setCurrentWidget(self.intro_page)

        self.statusBar().showMessage("Returned to home", 1300)

        QTimer.singleShot(200, self.start_intro_transition)



    def on_settings_clicked(self):

        QMessageBox.information(

            self,

            "Workspace Settings",

            "UI Theme: Luxe Dark\nStreaming: Enabled\nPanels: Animated\nTip: Use selectors in top bar for persona and prompt control.",

        )



    def on_model_changed(self, idx):

        selected = self.cmb_model.currentText()

        if idx != 0:

            self.statusBar().showMessage(f"{selected} not loaded yet. Using current local model.", 2200)

            self.cmb_model.blockSignals(True)

            self.cmb_model.setCurrentIndex(0)

            self.cmb_model.blockSignals(False)

            selected = self.cmb_model.currentText()

        self.info_model.setText(f"Model: {selected}")



    def on_new_chat_clicked(self):

        self._create_new_session()



    def on_toggle_left_panel(self):

        self._animate_panel(self.left_panel, self.LEFT_W, expand=not self.left_expanded)

        self.left_expanded = not self.left_expanded

        self.btn_left_toggle.setText("☰" if self.left_expanded else "⇥")

        self.btn_left_toggle_header.setText("☰" if self.left_expanded else "⇥")



    def on_toggle_right_panel(self):

        self._animate_panel(self.right_panel, self.RIGHT_W, expand=not self.right_expanded)

        self.right_expanded = not self.right_expanded

        self.btn_right_toggle.setText("⇤" if self.right_expanded else "⇥")



    def on_toggle_sessions_section(self):

        show = not self.sessions_box.isVisible()

        self.sessions_box.setVisible(show)

        self.section_sessions_btn.setText(("▾  " if show else "▸  ") + "Conversations")



    def on_toggle_files_section(self):

        show = not self.files_box.isVisible()

        self.files_box.setVisible(show)

        self.section_files_btn.setText(("▾  " if show else "▸  ") + "Uploaded Files")



    def on_toggle_controls_section(self):

        show = not self.controls_box.isVisible()

        self.controls_box.setVisible(show)

        self.section_controls_btn.setText(("▾  " if show else "▸  ") + "Runtime Controls")



    def on_upload_clicked(self):

        paths, _ = QFileDialog.getOpenFileNames(self, "Upload context files", "", "Text Files (*.txt *.md *.py *.json *.csv);;All Files (*)")

        if not paths:

            return



        added = 0

        for path in paths:

            try:

                with open(path, "r", encoding="utf-8", errors="replace") as f:

                    content = f.read()

            except Exception as e:

                QMessageBox.warning(self, "Read error", f"Could not read {path}\n{e}")

                continue

            name = os.path.basename(path)

            self.files_content[name] = content

            added += 1



        self._refresh_file_list()

        self.statusBar().showMessage(f"Added {added} file(s)", 2000)



    def on_file_selected(self, item):

        name = item.text()

        self.file_preview.setPlainText(self.files_content.get(name, ""))



    def on_session_selected(self, item):

        idx = item.data(Qt.UserRole)

        if idx is None or idx < 0 or idx >= len(self.sessions):

            return

        self.current_session_index = idx

        self._load_current_session_into_chat()



    def on_send_clicked(self):

        if self.worker and self.worker.isRunning():

            return

        text = self.input_box.text().strip()

        if not text:

            return



        if self.current_session_index < 0:

            self._create_new_session()



        persona_name = self.cmb_persona.currentText()

        prompt_name = self.cmb_system_prompt.currentText()

        persona_prompt = PERSONA_CHOICES.get(persona_name, "")

        system_prompt = SYSTEM_PROMPTS.get(prompt_name, SYSTEM_PROMPTS["Default"])

        self.info_persona.setText(f"Persona: {persona_name}")

        self.info_prompt.setText(f"System: {prompt_name}")



        session = self.sessions[self.current_session_index]

        if len(session["history"]) == 0:

            session["title"] = text[:46] + ("..." if len(text) > 46 else "")

            self._refresh_session_list()

            self.title_lbl.setText(session["title"] or "New Conversation")



        self.input_box.clear()

        self._add_bubble("user", text)

        session["history"].append({"role": "user", "content": text})



        system_seed = [

            {"role": "user", "content": f"{system_prompt}\n\nPersona mode: {persona_prompt}"},

            {"role": "model", "content": "Understood."},

        ]



        ephemeral = list(system_seed)

        ephemeral.extend(session["history"][:-1])



        context = self._build_context_block()

        aug = f"[REF]\n{context}\n[END]\n\nUser Question: {text}" if context else text

        ephemeral.append({"role": "user", "content": aug})



        self.current_reply = ""

        self.current_bubble = self._add_bubble("model", "Thinking...")



        self.worker = GenerationWorker(self.model, self.tok, ephemeral)

        self.worker.chunk.connect(self.on_stream_chunk)

        self.worker.finished_ok.connect(self.on_stream_finished)

        self.worker.failed.connect(self.on_stream_error)

        self.worker.start()



    def on_stream_chunk(self, chunk):

        self.current_reply += chunk

        if self.current_bubble:

            self.current_bubble.update_text(self.current_reply)

            self._update_bubble_widths()

            self._scroll_to_bottom()



    def on_stream_finished(self):

        if self.current_session_index >= 0 and self.current_reply.strip():

            self.sessions[self.current_session_index]["history"].append(

                {"role": "model", "content": self.current_reply.strip()}

            )

        self.current_reply = ""

        self.current_bubble = None



    def on_stream_error(self, err):

        if self.current_bubble:

            self.current_bubble.update_text(f"⚠ Streaming error: {err}")

        self.current_reply = ""

        self.current_bubble = None



    def showEvent(self, event):

        super().showEvent(event)

        self._apply_windows_dark_titlebar()

        self._update_bubble_widths()



    def resizeEvent(self, event):

        super().resizeEvent(event)

        self._update_bubble_widths()



    # ── Helpers ───────────────────────────────────────────────────────────────

    def _animate_panel(self, panel, expanded_width, expand=True):

        end_w = expanded_width if expand else 0

        self._anim = QPropertyAnimation(panel, b"maximumWidth")

        self._anim.setDuration(220)

        self._anim.setStartValue(panel.maximumWidth())

        self._anim.setEndValue(end_w)

        self._anim.setEasingCurve(QEasingCurve.InOutCubic)

        self._anim.start()



    def _apply_windows_dark_titlebar(self):

        if os.name != "nt":

            return

        try:

            import ctypes

            hwnd = int(self.winId())

            value = ctypes.c_int(1)

            ctypes.windll.dwmapi.DwmSetWindowAttribute(

                ctypes.c_void_p(hwnd),

                ctypes.c_uint(20),

                ctypes.byref(value),

                ctypes.sizeof(value),

            )

        except Exception:

            pass



    def _update_bubble_widths(self):

        if not hasattr(self, "chat_scroll"):

            return

        target = max(520, int(self.chat_scroll.viewport().width() * 0.86))

        for bubble in self.chat_host.findChildren(ChatBubble):

            bubble.setMaximumWidth(target)



    def _build_context_block(self):

        blocks = []

        for name in sorted(self.files_content.keys()):

            txt = self.files_content.get(name, "").strip()

            if txt:

                blocks.append(f"--- FILE: {name} ---\n{txt}\n--- END: {name} ---")

        return "\n\n".join(blocks)



    def _create_new_session(self, initial=False):

        self.sessions.append({"title": "New Conversation", "history": []})

        self.current_session_index = len(self.sessions) - 1

        self._refresh_session_list()

        self._clear_chat_view()

        self.title_lbl.setText("New Conversation")

        if not initial:

            self.statusBar().showMessage("Started new conversation", 1500)



    def _refresh_session_list(self):

        self.session_list.clear()

        for idx, session in enumerate(self.sessions):

            title = session.get("title") or f"Conversation {idx + 1}"

            item = QListWidgetItem(title)

            item.setData(Qt.UserRole, idx)

            self.session_list.addItem(item)

        if 0 <= self.current_session_index < self.session_list.count():

            self.session_list.setCurrentRow(self.current_session_index)



    def _refresh_file_list(self):

        self.file_list.clear()

        for name in sorted(self.files_content.keys()):

            self.file_list.addItem(name)



    def _load_current_session_into_chat(self):

        self._clear_chat_view()

        if self.current_session_index < 0:

            return

        session = self.sessions[self.current_session_index]

        self.title_lbl.setText(session.get("title", "Conversation"))

        for msg in session["history"]:

            self._add_bubble(msg.get("role", "model"), msg.get("content", ""))



    def _add_bubble(self, role, text):

        if self.chat_vbox.count() > 0:

            self.chat_vbox.takeAt(self.chat_vbox.count() - 1)



        bubble = ChatBubble(role, text)

        bubble.setMaximumWidth(max(520, int(self.chat_scroll.viewport().width() * 0.86)))

        row = QHBoxLayout()

        row.setContentsMargins(0, 0, 0, 0)



        if role == "user":

            row.addStretch(1)

            row.addWidget(bubble, 0, Qt.AlignRight)

        else:

            row.addWidget(bubble, 0, Qt.AlignLeft)

            row.addStretch(1)



        holder = QWidget()

        holder.setLayout(row)

        self.chat_vbox.addWidget(holder)

        self.chat_vbox.addStretch(1)

        self._scroll_to_bottom()

        return bubble



    def _clear_chat_view(self):

        while self.chat_vbox.count():

            item = self.chat_vbox.takeAt(0)

            widget = item.widget()

            if widget is not None:

                widget.deleteLater()

        self.chat_vbox.addStretch(1)



    def _scroll_to_bottom(self):

        sb = self.chat_scroll.verticalScrollBar()

        sb.setValue(sb.maximum())



def launch_gemma_premium_ui(mdl, tok):

    try:

        from IPython import get_ipython

        ip = get_ipython()

        if ip is not None:

            ip.run_line_magic("gui", "qt")

    except Exception:

        pass



    app = QApplication.instance()

    owns_app = False

    if app is None:

        app = QApplication([])

        owns_app = True



    win = PremiumGemmaWindow(mdl, tok)

    win.show()



    if owns_app:

        app.exec_()



    return win



# ── LAUNCH ────────────────────────────────────────────────────────────────────

try:

    premium_window = launch_gemma_premium_ui(model, tokenizer)

    print("✅ Luxe Premium PyQt5 UI launched with selectors, branding, and dynamic workspace.")

except Exception as e:

    print(f"❌ Failed to launch Premium UI: {e}")


✅ Luxe Premium PyQt5 UI launched with selectors, branding, and dynamic workspace.


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# UI_V10 — NOTEBOOK TYPOGRAPHY CHAT (refined)
# - Clean notebook-style chat (no blot/bubble blocks)
# - Dual themes: Navy+Orange (Dark) / OffWhite+Brown (Light)
# - 125% visual scale + constrained content width for larger appearance
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
from threading import Thread

from PyQt5.QtCore import Qt, QThread, pyqtSignal
from PyQt5.QtGui import QFont, QFontDatabase, QIcon, QPixmap, QTextOption
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QFrame, QLabel, QPushButton, QLineEdit,

    QTextBrowser, QTextEdit, QListWidget, QListWidgetItem, QFileDialog, QMessageBox,
    QHBoxLayout, QVBoxLayout, QSizePolicy, QScrollArea, QComboBox
)

from transformers import TextIteratorStreamer

UI_SCALE = 1.25
LOGO_PATH_UI10 = r"C:\Users\divyam.tewary\OneDrive - InterGlobe Aviation Limited\Desktop\IndiGo_logo.png"

THEMES_UI10 = {
    "Navy + Orange (Dark)": {
        "bg": "#081120",
        "bg2": "#0D1B33",
        "panel": "#0F1E39",
        "panel_soft": "#132547",
        "border": "#2A426F",
        "text": "#EAF0FF",
        "muted": "#9FB0D9",
        "accent": "#FF8A2A",
        "accent_hover": "#FF9B4E",
        "entry_user": "#1A2F58",
        "entry_model": "#122647",
        "input_bg": "#0F1E39",
    },
    "Off White + Brown (Light)": {
        "bg": "#F8F4EC",
        "bg2": "#F1E7DA",
        "panel": "#FFFDF8",
        "panel_soft": "#F7EFE3",
        "border": "#C9B197",
        "text": "#2F241B",
        "muted": "#6A5848",
        "accent": "#8C5A2B",
        "accent_hover": "#A06A37",
        "entry_user": "#F4E7D9",
        "entry_model": "#FAF3E9",
        "input_bg": "#FFFDF8",
    },
}

MODEL_OPTIONS_UI10 = [
    "Gemma 3 1B-IT (Loaded)",
    "Gemma 3 4B-IT (Placeholder)",
    "Gemma 3 12B-IT (Placeholder)",
]

PERSONAS_UI10 = {
    "Balanced": "Be concise, accurate, and practical.",
    "Coder": "You are a senior coding assistant. Give production-ready solutions.",
    "Analyst": "Provide structured analysis with assumptions and trade-offs.",
    "Creative": "Be imaginative, clear, and user-centric.",
}

SYSTEM_PROMPTS_UI10 = {
    "Default": "You are a helpful and concise AI assistant. Format clearly using headings and bullet points where useful.",
    "Strict Technical": "Be precise, technical, and avoid vague statements.",
    "Tutor": "Teach in simple steps with concrete examples.",
    "Exec": "Provide concise executive summaries with actions.",
}


def _font_ui10(base=10):
    families = set(QFontDatabase().families())
    for name in ["Segoe UI Variable", "Inter", "Segoe UI", "Arial"]:
        if name in families:
            return QFont(name, max(9, int(base * UI_SCALE)))
    return QFont("Sans Serif", max(9, int(base * UI_SCALE)))


class Ui10Button(QPushButton):
    def __init__(self, text, callback=None, accent=False, compact=False):
        super().__init__(text)
        self._accent = accent
        self._compact = compact
        self.setCursor(Qt.PointingHandCursor)
        self.setMinimumHeight(36)
        if callback:
            self.clicked.connect(callback)

    def apply_theme(self, theme):
        pad = "6px 10px" if self._compact else "8px 12px"
        if self._accent:
            self.setStyleSheet(
                f"""
                QPushButton {{
                    background: {theme['accent']};
                    color: white;
                    border: 1px solid {theme['accent']};
                    border-radius: 10px;
                    padding: {pad};
                    font-weight: 700;
                }}
                QPushButton:hover {{ background: {theme['accent_hover']}; }}
                """
            )
        else:
            self.setStyleSheet(
                f"""
                QPushButton {{
                    background: {theme['panel']};
                    color: {theme['text']};
                    border: 1px solid {theme['border']};
                    border-radius: 10px;
                    padding: {pad};
                    font-weight: 600;
                    text-align: left;
                }}
                QPushButton:hover {{ background: {theme['panel_soft']}; }}
                """
            )


class NotebookEntryV10(QFrame):
    def __init__(self, role, text=""):
        super().__init__()
        self.role = role
        self.text = text

        self.role_lbl = QLabel("You" if role == "user" else "IndiGo AI")
        self.role_lbl.setSizePolicy(QSizePolicy.Preferred, QSizePolicy.Fixed)

        self.body = QTextBrowser()
        self.body.setOpenExternalLinks(True)
        self.body.setFrameShape(QFrame.NoFrame)
        self.body.setVerticalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.body.setHorizontalScrollBarPolicy(Qt.ScrollBarAlwaysOff)
        self.body.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Minimum)

        opt = self.body.document().defaultTextOption()
        opt.setWrapMode(QTextOption.WrapAtWordBoundaryOrAnywhere)
        self.body.document().setDefaultTextOption(opt)

        self.divider = QFrame()
        self.divider.setFrameShape(QFrame.HLine)
        self.divider.setFixedHeight(1)

        lay = QVBoxLayout(self)
        lay.setContentsMargins(10, 10, 10, 10)
        lay.setSpacing(6)
        lay.addWidget(self.role_lbl)
        lay.addWidget(self.body)
        lay.addWidget(self.divider)

        self.update_text(text)

    def _md(self, s):
        s = (s or "").replace("\r\n", "\n")
        return re.sub(r"\n{3,}", "\n\n", s).strip() or " "

    def apply_theme(self, theme):
        block_bg = theme["entry_user"] if self.role == "user" else theme["entry_model"]
        self.setStyleSheet(
            f"QFrame {{ background: {block_bg}; border: 1px solid {theme['border']}; border-radius: 10px; }}"
        )
        self.role_lbl.setStyleSheet(
            f"font-size: 13px; font-weight: 700; color: {theme['muted']}; background: transparent; border: none;"
        )
        self.body.setStyleSheet(
            f"QTextBrowser {{ background: transparent; border: none; color: {theme['text']}; font-size: 16px; line-height: 1.6; }}"
        )
        self.divider.setStyleSheet(f"background: {theme['border']}; border: none;")

    def update_text(self, text):
        self.text = text
        self.body.setMarkdown(self._md(text))
        self.body.document().adjustSize()
        h = int(self.body.document().size().height()) + 8
        self.body.setMinimumHeight(max(36, h))
        self.body.setMaximumHeight(max(36, h + 8))

    def sync_width(self, column_width):
        column_width = max(600, int(column_width))
        self.setMaximumWidth(column_width)
        text_w = max(540, column_width - 28)
        self.body.setFixedWidth(text_w)
        self.body.document().setTextWidth(max(500, text_w - 10))
        self.body.document().adjustSize()
        h = int(self.body.document().size().height()) + 8
        self.body.setMinimumHeight(max(36, h))
        self.body.setMaximumHeight(max(36, h + 8))


class WorkerV10(QThread):
    chunk = pyqtSignal(str)
    finished_ok = pyqtSignal()
    failed = pyqtSignal(str)

    def __init__(self, mdl, tok, prompt_history):
        super().__init__()
        self.model = mdl
        self.tok = tok
        self.prompt_history = prompt_history

    def run(self):
        try:
            prompt = self.tok.apply_chat_template(self.prompt_history, tokenize=False, add_generation_prompt=True)
            inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)
            streamer = TextIteratorStreamer(self.tok, timeout=None, skip_prompt=True, skip_special_tokens=True)
            terms = [self.tok.eos_token_id, self.tok.convert_tokens_to_ids("<end_of_turn>")]

            def _generate():
                self.model.generate(
                    **inputs,
                    streamer=streamer,
                    max_new_tokens=768,
                    do_sample=False,
                    eos_token_id=terms,
                    pad_token_id=self.tok.eos_token_id,
                )

            t = Thread(target=_generate, daemon=True)
            t.start()
            for token in streamer:
                token = token.replace("<end_of_turn>", "")
                if token:
                    self.chunk.emit(token)
            t.join()
            self.finished_ok.emit()
        except Exception as e:
            self.failed.emit(str(e))


class PremiumGemmaWindowV10(QMainWindow):
    LEFT_W = 250
    RIGHT_W = 340
    CHAT_MAX_COLUMN = 980

    def __init__(self, mdl, tok):
        super().__init__()
        self.model = mdl
        self.tok = tok

        self.worker = None
        self.current_reply = ""
        self.current_entry = None

        self.sessions = []
        self.current_session_index = -1
        self.files_content = {}
        self.left_expanded = True
        self.right_expanded = True

        self.theme_name = "Navy + Orange (Dark)"

        self.setWindowTitle("IndiGo AI Workspace — UI10")
        self.resize(int(1460 * UI_SCALE), int(920 * UI_SCALE))
        QApplication.instance().setFont(_font_ui10(11))

        self._build_ui()
        self._apply_windows_dark_titlebar()
        self._apply_theme(self.theme_name)

        self._create_new_session(initial=True)
        self.statusBar().showMessage("UI10 ready")

    def _combo_style(self, theme):
        return (
            f"""
            QComboBox {{
                background:{theme['panel']};
                color:{theme['text']};
                border:1px solid {theme['border']};
                border-radius:10px;
                padding:6px 10px;
                min-width:170px;
                font-weight:600;
            }}
            QComboBox QAbstractItemView {{
                background:{theme['panel']};
                color:{theme['text']};
                border:1px solid {theme['border']};
                selection-background-color:{theme['panel_soft']};
            }}
            """
        )

    def _build_ui(self):
        root = QWidget()
        self.setCentralWidget(root)
        hroot = QHBoxLayout(root)
        hroot.setContentsMargins(0, 0, 0, 0)
        hroot.setSpacing(0)

        self.left_panel = QFrame()
        self.left_panel.setMaximumWidth(self.LEFT_W)
        l = QVBoxLayout(self.left_panel)
        l.setContentsMargins(12, 12, 12, 12)
        l.setSpacing(8)

        row = QHBoxLayout()
        self.btn_left_toggle = Ui10Button("☰", self.on_toggle_left_panel, compact=True)
        self.btn_left_toggle.setFixedWidth(42)
        self.btn_home = Ui10Button("Home", self.on_home_clicked, compact=True)
        if os.path.exists(LOGO_PATH_UI10):
            self.btn_home.setText("")
            self.btn_home.setIcon(QIcon(LOGO_PATH_UI10))
            self.btn_home.setIconSize(QPixmap(LOGO_PATH_UI10).scaledToHeight(28, Qt.SmoothTransformation).size())
        row.addWidget(self.btn_left_toggle)
        row.addWidget(self.btn_home, 1)
        l.addLayout(row)

        self.left_title = QLabel("Conversations")
        l.addWidget(self.left_title)
        self.btn_new_chat = Ui10Button("＋ New Chat", self.on_new_chat_clicked, accent=True)
        l.addWidget(self.btn_new_chat)

        self.session_list = QListWidget()
        self.session_list.itemClicked.connect(self.on_session_selected)
        l.addWidget(self.session_list, 1)

        self.center_panel = QFrame()
        c = QVBoxLayout(self.center_panel)
        c.setContentsMargins(14, 12, 14, 12)
        c.setSpacing(10)

        self.header = QFrame()
        hh = QHBoxLayout(self.header)
        hh.setContentsMargins(12, 10, 12, 10)
        hh.setSpacing(10)

        self.logo_lbl = QLabel()
        self.logo_lbl.setFixedHeight(30)
        if os.path.exists(LOGO_PATH_UI10):
            pix = QPixmap(LOGO_PATH_UI10)
            self.logo_lbl.setPixmap(pix.scaled(120, 30, Qt.KeepAspectRatio, Qt.SmoothTransformation))
        else:
            self.logo_lbl.setText("INDIGO")

        self.lbl_title = QLabel("Notebook Conversation")

        self.cmb_model = QComboBox(); self.cmb_model.addItems(MODEL_OPTIONS_UI10); self.cmb_model.currentIndexChanged.connect(self.on_model_changed)
        self.cmb_persona = QComboBox(); self.cmb_persona.addItems(list(PERSONAS_UI10.keys()))
        self.cmb_system = QComboBox(); self.cmb_system.addItems(list(SYSTEM_PROMPTS_UI10.keys()))
        self.cmb_theme = QComboBox(); self.cmb_theme.addItems(list(THEMES_UI10.keys())); self.cmb_theme.currentTextChanged.connect(self.on_theme_changed)

        self.btn_toggle_right = Ui10Button("Controls", self.on_toggle_right_panel, compact=True)

        hh.addWidget(self.logo_lbl)
        hh.addWidget(self.lbl_title)
        hh.addStretch(1)
        hh.addWidget(self.cmb_model)
        hh.addWidget(self.cmb_persona)
        hh.addWidget(self.cmb_system)
        hh.addWidget(self.cmb_theme)
        hh.addWidget(self.btn_toggle_right)

        self.chat_scroll = QScrollArea()
        self.chat_scroll.setWidgetResizable(True)
        self.chat_host = QWidget()
        self.chat_vbox = QVBoxLayout(self.chat_host)
        self.chat_vbox.setContentsMargins(16, 18, 16, 18)
        self.chat_vbox.setSpacing(10)
        self.chat_vbox.addStretch(1)
        self.chat_scroll.setWidget(self.chat_host)

        self.input_wrap = QFrame()
        ih = QHBoxLayout(self.input_wrap)
        ih.setContentsMargins(12, 10, 12, 10)
        ih.setSpacing(8)

        self.input_box = QLineEdit()
        self.input_box.setPlaceholderText("Write your prompt…")
        self.input_box.returnPressed.connect(self.on_send_clicked)

        self.btn_upload = Ui10Button("Upload", self.on_upload_clicked, compact=True)
        self.btn_send = Ui10Button("Send", self.on_send_clicked, compact=True, accent=True)
        self.btn_send.setFixedWidth(90)

        ih.addWidget(self.input_box, 1)
        ih.addWidget(self.btn_upload)
        ih.addWidget(self.btn_send)

        c.addWidget(self.header)
        c.addWidget(self.chat_scroll, 1)
        c.addWidget(self.input_wrap)

        self.right_panel = QFrame()
        self.right_panel.setMaximumWidth(self.RIGHT_W)
        r = QVBoxLayout(self.right_panel)
        r.setContentsMargins(12, 12, 12, 12)
        r.setSpacing(8)

        top = QHBoxLayout()
        self.right_title = QLabel("Controls & Context")
        self.btn_right_toggle = Ui10Button("⇤", self.on_toggle_right_panel, compact=True)
        self.btn_right_toggle.setFixedWidth(42)
        top.addWidget(self.right_title); top.addStretch(1); top.addWidget(self.btn_right_toggle)

        self.file_list = QListWidget(); self.file_list.itemClicked.connect(self.on_file_selected)
        self.file_preview = QTextEdit(); self.file_preview.setReadOnly(True); self.file_preview.setPlaceholderText("File preview")
        self.info_runtime = QLabel("Model: Gemma 3 1B-IT\nPersona: Balanced\nSystem: Default")

        r.addLayout(top)
        r.addWidget(self.file_list, 1)
        r.addWidget(self.file_preview, 1)
        r.addWidget(self.info_runtime)

        hroot.addWidget(self.left_panel)
        hroot.addWidget(self.center_panel, 1)
        hroot.addWidget(self.right_panel)

    def _apply_theme(self, theme_name):
        self.theme_name = theme_name
        t = THEMES_UI10[theme_name]

        self.setStyleSheet(
            f"QMainWindow {{ background: {t['bg']}; color: {t['text']}; }}"
            f"QScrollBar:vertical {{ width: 12px; background: {t['panel']}; }}"
            f"QScrollBar::handle:vertical {{ background: {t['border']}; border-radius: 6px; min-height: 24px; }}"
        )

        self.left_panel.setStyleSheet(f"background:{t['panel']}; border-right:1px solid {t['border']};")
        self.center_panel.setStyleSheet(f"background:{t['bg2']};")
        self.right_panel.setStyleSheet(f"background:{t['panel']}; border-left:1px solid {t['border']};")

        self.header.setStyleSheet(f"QFrame {{ background:{t['panel']}; border:1px solid {t['border']}; border-radius:12px; }}")
        self.input_wrap.setStyleSheet(f"QFrame {{ background:{t['input_bg']}; border:1px solid {t['border']}; border-radius:12px; }}")
        self.chat_scroll.setStyleSheet(f"QScrollArea {{ border:1px solid {t['border']}; border-radius:12px; background:{t['bg2']}; }}")

        self.left_title.setStyleSheet(f"font-size:12px; font-weight:700; color:{t['muted']};")
        self.lbl_title.setStyleSheet(f"font-size:18px; font-weight:800; color:{t['text']};")
        self.right_title.setStyleSheet(f"font-size:13px; font-weight:800; color:{t['text']};")
        self.info_runtime.setStyleSheet(f"color:{t['muted']}; line-height:1.5;")

        self.session_list.setStyleSheet(
            f"QListWidget {{ background:{t['bg2']}; color:{t['text']}; border:1px solid {t['border']}; border-radius:10px; padding:4px; }}"
            f"QListWidget::item {{ padding:8px; border-radius:8px; }}"
            f"QListWidget::item:selected {{ background:{t['panel_soft']}; }}"
        )
        self.file_list.setStyleSheet(
            f"QListWidget {{ background:{t['bg2']}; color:{t['text']}; border:1px solid {t['border']}; border-radius:10px; padding:4px; }}"
            f"QListWidget::item {{ padding:8px; border-radius:8px; }}"
            f"QListWidget::item:selected {{ background:{t['panel_soft']}; }}"
        )
        self.file_preview.setStyleSheet(
            f"QTextEdit {{ background:{t['bg2']}; color:{t['text']}; border:1px solid {t['border']}; border-radius:10px; padding:8px; }}"
        )
        self.input_box.setStyleSheet(
            f"QLineEdit {{ background:transparent; border:none; color:{t['text']}; font-size:15px; }}"
        )

        combo_css = self._combo_style(t)
        self.cmb_model.setStyleSheet(combo_css)
        self.cmb_persona.setStyleSheet(combo_css)
        self.cmb_system.setStyleSheet(combo_css)
        self.cmb_theme.setStyleSheet(combo_css)

        for b in [self.btn_left_toggle, self.btn_home, self.btn_toggle_right, self.btn_right_toggle, self.btn_upload]:
            b.apply_theme(t)
        for b in [self.btn_new_chat, self.btn_send]:
            b.apply_theme(t)

        for entry in self.chat_host.findChildren(NotebookEntryV10):
            entry.apply_theme(t)
            entry.sync_width(self._chat_column_width())

        self.statusBar().setStyleSheet(f"background:{t['panel']}; color:{t['muted']}; border-top:1px solid {t['border']};")

    def _chat_column_width(self):
        viewport_w = self.chat_scroll.viewport().width()
        return min(int(viewport_w * 0.96), int(self.CHAT_MAX_COLUMN * UI_SCALE))

    def _apply_windows_dark_titlebar(self):
        if os.name != "nt":
            return
        try:
            import ctypes
            hwnd = int(self.winId())
            value = ctypes.c_int(1 if "Dark" in self.theme_name else 0)
            ctypes.windll.dwmapi.DwmSetWindowAttribute(
                ctypes.c_void_p(hwnd), ctypes.c_uint(20), ctypes.byref(value), ctypes.sizeof(value)
            )
        except Exception:
            pass

    # actions
    def on_home_clicked(self):
        self.statusBar().showMessage("Home clicked", 1200)

    def on_theme_changed(self, name):
        self._apply_theme(name)
        self._apply_windows_dark_titlebar()

    def on_model_changed(self, idx):
        if idx != 0:
            picked = self.cmb_model.currentText()
            self.statusBar().showMessage(f"{picked} not loaded; using Gemma 3 1B-IT.", 2000)
            self.cmb_model.blockSignals(True)
            self.cmb_model.setCurrentIndex(0)
            self.cmb_model.blockSignals(False)
        self._update_runtime_info()

    def on_new_chat_clicked(self):
        self._create_new_session()

    def on_toggle_left_panel(self):
        self.left_expanded = not self.left_expanded
        self.left_panel.setMaximumWidth(self.LEFT_W if self.left_expanded else 0)
        self.btn_left_toggle.setText("☰" if self.left_expanded else "⇥")
        self.btn_left_toggle_header = getattr(self, "btn_left_toggle_header", None)

    def on_toggle_right_panel(self):
        self.right_expanded = not self.right_expanded
        self.right_panel.setMaximumWidth(self.RIGHT_W if self.right_expanded else 0)
        self.btn_right_toggle.setText("⇤" if self.right_expanded else "⇥")

    def on_upload_clicked(self):
        paths, _ = QFileDialog.getOpenFileNames(self, "Upload context files", "", "Text Files (*.txt *.md *.py *.json *.csv);;All Files (*)")
        if not paths:
            return
        added = 0
        for path in paths:
            try:
                with open(path, "r", encoding="utf-8", errors="replace") as f:
                    content = f.read()
            except Exception as e:
                QMessageBox.warning(self, "Read error", f"Could not read {path}\n{e}")
                continue
            self.files_content[os.path.basename(path)] = content
            added += 1
        self._refresh_file_list()
        self.statusBar().showMessage(f"Added {added} file(s)", 1800)

    def on_file_selected(self, item):
        self.file_preview.setPlainText(self.files_content.get(item.text(), ""))

    def on_session_selected(self, item):
        idx = item.data(Qt.UserRole)
        if idx is None or idx < 0 or idx >= len(self.sessions):
            return
        self.current_session_index = idx
        self._load_current_session_into_chat()

    def on_send_clicked(self):
        if self.worker and self.worker.isRunning():
            return
        text = self.input_box.text().strip()
        if not text:
            return

        if self.current_session_index < 0:
            self._create_new_session()
        self._update_runtime_info()

        persona_name = self.cmb_persona.currentText()
        sys_name = self.cmb_system.currentText()
        persona_prompt = PERSONAS_UI10.get(persona_name, "")
        sys_prompt = SYSTEM_PROMPTS_UI10.get(sys_name, SYSTEM_PROMPTS_UI10["Default"])

        session = self.sessions[self.current_session_index]
        if len(session["history"]) == 0:
            session["title"] = text[:56] + ("..." if len(text) > 56 else "")
            self._refresh_session_list()
            self.lbl_title.setText(session["title"])

        self.input_box.clear()
        self._add_entry("user", text)
        session["history"].append({"role": "user", "content": text})

        seed = [
            {"role": "user", "content": f"{sys_prompt}\n\nPersona: {persona_prompt}"},
            {"role": "model", "content": "Understood."},
        ]
        ephemeral = list(seed)
        ephemeral.extend(session["history"][:-1])

        context = self._build_context_block()
        aug = f"[REF]\n{context}\n[END]\n\nUser Question: {text}" if context else text
        ephemeral.append({"role": "user", "content": aug})

        self.current_reply = ""
        self.current_entry = self._add_entry("model", "Thinking…")
        self.worker = WorkerV10(self.model, self.tok, ephemeral)
        self.worker.chunk.connect(self.on_stream_chunk)
        self.worker.finished_ok.connect(self.on_stream_finished)
        self.worker.failed.connect(self.on_stream_error)
        self.worker.start()

    def on_stream_chunk(self, chunk):
        self.current_reply += chunk
        if self.current_entry:
            self.current_entry.update_text(self.current_reply)
            self._sync_entries()
            self._scroll_to_bottom()

    def on_stream_finished(self):
        if self.current_session_index >= 0 and self.current_reply.strip():
            self.sessions[self.current_session_index]["history"].append({"role": "model", "content": self.current_reply.strip()})
        self.current_reply = ""
        self.current_entry = None

    def on_stream_error(self, err):
        if self.current_entry:
            self.current_entry.update_text(f"⚠ Streaming error: {err}")
            self._sync_entries()
        self.current_reply = ""
        self.current_entry = None

    # helpers
    def _update_runtime_info(self):
        self.info_runtime.setText(
            f"Model: {self.cmb_model.currentText()}\n"
            f"Persona: {self.cmb_persona.currentText()}\n"
            f"System: {self.cmb_system.currentText()}\n"
            f"Theme: {self.cmb_theme.currentText()}"
        )

    def _sync_entries(self):
        w = self._chat_column_width()
        for entry in self.chat_host.findChildren(NotebookEntryV10):
            entry.sync_width(w)

    def resizeEvent(self, event):
        super().resizeEvent(event)
        self._sync_entries()

    def showEvent(self, event):
        super().showEvent(event)
        self._sync_entries()

    def _build_context_block(self):
        blocks = []
        for name in sorted(self.files_content.keys()):
            txt = self.files_content.get(name, "").strip()
            if txt:
                blocks.append(f"--- FILE: {name} ---\n{txt}\n--- END: {name} ---")
        return "\n\n".join(blocks)

    def _create_new_session(self, initial=False):
        self.sessions.append({"title": "New Conversation", "history": []})
        self.current_session_index = len(self.sessions) - 1
        self._refresh_session_list()
        self._clear_chat_view()
        self.lbl_title.setText("Notebook Conversation")
        if not initial:
            self.statusBar().showMessage("Started new conversation", 1200)

    def _refresh_session_list(self):
        self.session_list.clear()
        for idx, session in enumerate(self.sessions):
            item = QListWidgetItem(session.get("title") or f"Conversation {idx + 1}")
            item.setData(Qt.UserRole, idx)
            self.session_list.addItem(item)
        if 0 <= self.current_session_index < self.session_list.count():
            self.session_list.setCurrentRow(self.current_session_index)

    def _refresh_file_list(self):
        self.file_list.clear()
        for name in sorted(self.files_content.keys()):
            self.file_list.addItem(name)

    def _load_current_session_into_chat(self):
        self._clear_chat_view()
        if self.current_session_index < 0:
            return
        session = self.sessions[self.current_session_index]
        self.lbl_title.setText(session.get("title", "Conversation"))
        for msg in session["history"]:
            self._add_entry(msg.get("role", "model"), msg.get("content", ""))

    def _add_entry(self, role, text):
        if self.chat_vbox.count() > 0:
            self.chat_vbox.takeAt(self.chat_vbox.count() - 1)

        entry = NotebookEntryV10(role, text)
        entry.apply_theme(THEMES_UI10[self.theme_name])

        row = QHBoxLayout()
        row.setContentsMargins(0, 0, 0, 0)
        row.addStretch(1)
        row.addWidget(entry, 0, Qt.AlignTop)
        row.addStretch(1)

        holder = QWidget()
        holder.setLayout(row)
        self.chat_vbox.addWidget(holder)
        self.chat_vbox.addStretch(1)

        self._sync_entries()
        self._scroll_to_bottom()
        return entry

    def _clear_chat_view(self):
        while self.chat_vbox.count():
            item = self.chat_vbox.takeAt(0)
            widget = item.widget()
            if widget is not None:
                widget.deleteLater()
        self.chat_vbox.addStretch(1)

    def _scroll_to_bottom(self):
        sb = self.chat_scroll.verticalScrollBar()
        sb.setValue(sb.maximum())


def launch_gemma_ui10(mdl, tok):
    os.environ.setdefault("QT_SCALE_FACTOR", "1.25")
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("gui", "qt")
    except Exception:
        pass

    app = QApplication.instance()
    owns_app = False
    if app is None:
        app = QApplication([])
        owns_app = True

    win = PremiumGemmaWindowV10(mdl, tok)
    win.show()

    if owns_app:
        app.exec_()
    return win

# ── LAUNCH UI10 ───────────────────────────────────────────────────────────────
try:
    premium_window_ui10 = launch_gemma_ui10(model, tokenizer)
    print("✅ UI10 launched: notebook typography chat + dual clean themes.")
except Exception as e:
    print(f"❌ Failed to launch UI10: {e}")


✅ UI10 launched: notebook typography chat + dual clean themes.
